In [ ]:
"""
QUICKBITE PRODUCT ANALYTICS PLATFORM
Phase 2 - Notebook 9: LTV Prediction
==================================================================
Purpose: Calculate historical LTV, build predictive LTV models,
and analyze LTV:CAC ratios to optimize acquisition spending.

Key Questions:
1. What is the historical LTV of our users?
2. Can we predict future LTV using early behavior?
3. Which acquisition channels have the best LTV:CAC ratio?
4. How can we optimize acquisition spend based on LTV?

Author: Senior Product Analytics Team
Date: 2026-07-28
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
print("="*80)
print("QUICKBITE LTV PREDICTION")
print("="*80)

---------------------------------------------------------------------
1. LOAD CLEANED DATA
---------------------------------------------------------------------

In [ ]:
print("\n📂 Loading cleaned data...")

In [ ]:
users = pd.read_csv('../outputs/cleaned_data/users_cleaned.csv')
orders = pd.read_csv('../outputs/cleaned_data/orders_cleaned.csv')
order_items = pd.read_csv('../outputs/cleaned_data/order_items_cleaned.csv')
payments = pd.read_csv('../outputs/cleaned_data/payments_cleaned.csv')
rfm = pd.read_csv('../outputs/cleaned_data/rfm_segments.csv')
cities = pd.read_csv('../data/cities.csv')

In [ ]:
# Convert dates
users['signup_date'] = pd.to_datetime(users['signup_date'])
orders['order_placed_at'] = pd.to_datetime(orders['order_placed_at'])

In [ ]:
# Filter delivered orders
delivered_orders = orders[orders['order_status'] == 'delivered']

In [ ]:
print(f"✅ Loaded {len(users):,} users")
print(f"✅ Loaded {len(delivered_orders):,} delivered orders")

---------------------------------------------------------------------
2. CALCULATE HISTORICAL LTV
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("HISTORICAL LTV CALCULATION")
print("="*80)

In [ ]:
# Calculate LTV by user
ltv_data = delivered_orders.groupby('user_id').agg({
    'order_id': 'count',
    'total_amount': ['sum', 'mean', 'std'],
    'order_placed_at': ['min', 'max']
}).reset_index()

In [ ]:
ltv_data.columns = ['user_id', 'order_count', 'lifetime_value', 'avg_order_value', 
                    'std_order_value', 'first_order_date', 'last_order_date']

In [ ]:
# Calculate customer lifetime in days
ltv_data['customer_lifetime_days'] = (ltv_data['last_order_date'] - ltv_data['first_order_date']).dt.days
ltv_data['customer_lifetime_days'] = ltv_data['customer_lifetime_days'].fillna(0)

In [ ]:
# Calculate average days between orders
ltv_data['avg_days_between_orders'] = np.where(
    ltv_data['order_count'] > 1,
    ltv_data['customer_lifetime_days'] / (ltv_data['order_count'] - 1),
    0
)

In [ ]:
# Merge with user attributes
ltv_data = ltv_data.merge(
    users[['user_id', 'signup_date', 'acquisition_channel', 'is_premium_member', 
           'city_id', 'age_band', 'device_type']],
    on='user_id', how='left'
)

In [ ]:
# Add city names and tier
ltv_data = ltv_data.merge(cities[['city_id', 'city_name', 'tier']], on='city_id', how='left')

In [ ]:
# Add RFM segments
ltv_data = ltv_data.merge(
    rfm[['user_id', 'segment']], on='user_id', how='left'
)

In [ ]:
# Calculate signup to first order time
ltv_data['days_to_first_order'] = (ltv_data['first_order_date'] - ltv_data['signup_date']).dt.days

In [ ]:
print(f"✅ LTV calculated for {len(ltv_data):,} users")

In [ ]:
# Display LTV statistics
print("\n📊 LTV Statistics:")
print(ltv_data['lifetime_value'].describe())

In [ ]:
# LTV distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Historical LTV Analysis', fontsize=16, fontweight='bold')

In [ ]:
# LTV Histogram
ax = axes[0, 0]
ltv_data['lifetime_value'].hist(bins=50, ax=ax, color='#3498db', alpha=0.7, edgecolor='black')
ax.axvline(ltv_data['lifetime_value'].mean(), color='red', linestyle='--', 
           label=f"Mean: ₹{ltv_data['lifetime_value'].mean():.0f}")
ax.axvline(ltv_data['lifetime_value'].median(), color='green', linestyle='--',
           label=f"Median: ₹{ltv_data['lifetime_value'].median():.0f}")
ax.set_title('LTV Distribution')
ax.set_xlabel('Lifetime Value (₹)')
ax.set_ylabel('Number of Users')
ax.legend()

In [ ]:
# LTV by Acquisition Channel
ax = axes[0, 1]
channel_ltv = ltv_data.groupby('acquisition_channel')['lifetime_value'].mean().sort_values(ascending=False)
channel_ltv.plot(kind='bar', ax=ax, color='#2ecc71', alpha=0.7)
ax.set_title('Average LTV by Acquisition Channel')
ax.set_xlabel('Channel')
ax.set_ylabel('Average LTV (₹)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# LTV by Segment
ax = axes[1, 0]
segment_ltv = ltv_data.groupby('segment')['lifetime_value'].mean().sort_values(ascending=False)
segment_ltv.plot(kind='bar', ax=ax, color='#e67e22', alpha=0.7)
ax.set_title('Average LTV by RFM Segment')
ax.set_xlabel('Segment')
ax.set_ylabel('Average LTV (₹)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# LTV by City
ax = axes[1, 1]
city_ltv = ltv_data.groupby('city_name')['lifetime_value'].mean().sort_values(ascending=False)
city_ltv.head(10).plot(kind='barh', ax=ax, color='#9b59b6', alpha=0.7)
ax.set_title('Top 10 Cities by LTV')
ax.set_xlabel('Average LTV (₹)')

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/ltv_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
3. CAC (Customer Acquisition Cost) ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CAC ANALYSIS")
print("="*80)

In [ ]:
# Calculate CAC by channel
cac_data = users.groupby('acquisition_channel').agg({
    'user_id': 'count',
    'signup_channel_cost': 'sum'
}).reset_index()

In [ ]:
cac_data['cac'] = cac_data['signup_channel_cost'] / cac_data['user_id']

In [ ]:
# Merge with channel LTV
channel_data = cac_data.merge(
    ltv_data.groupby('acquisition_channel')['lifetime_value'].mean().reset_index(),
    on='acquisition_channel'
)

In [ ]:
channel_data['ltv_cac_ratio'] = channel_data['lifetime_value'] / channel_data['cac']

In [ ]:
print("\n📊 LTV:CAC Ratio by Channel:")
print(channel_data[['acquisition_channel', 'user_id', 'cac', 'lifetime_value', 'ltv_cac_ratio']].to_string(index=False))

In [ ]:
# Visualize LTV:CAC
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('LTV:CAC Analysis', fontsize=14, fontweight='bold')

In [ ]:
# LTV:CAC Ratio
ax = axes[0]
channel_data_sorted = channel_data.sort_values('ltv_cac_ratio', ascending=False)
bars = ax.bar(channel_data_sorted['acquisition_channel'], 
              channel_data_sorted['ltv_cac_ratio'],
              color=['#2ecc71' if x > 3 else '#f39c12' if x > 1 else '#e74c3c' 
                     for x in channel_data_sorted['ltv_cac_ratio']],
              alpha=0.7)
ax.axhline(y=3, color='green', linestyle='--', label='Good (3x)')
ax.axhline(y=1, color='red', linestyle='--', label='Breakeven (1x)')
ax.set_title('LTV:CAC Ratio by Channel')
ax.set_xlabel('Channel')
ax.set_ylabel('LTV:CAC Ratio')
ax.tick_params(axis='x', rotation=45)
ax.legend()

In [ ]:
for bar, ratio in zip(bars, channel_data_sorted['ltv_cac_ratio']):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.1,
            f'{ratio:.1f}x', ha='center', va='bottom', fontsize=9)

In [ ]:
# LTV vs CAC Scatter
ax = axes[1]
ax.scatter(channel_data['cac'], channel_data['lifetime_value'], 
           s=channel_data['user_id']/100, alpha=0.6, c=range(len(channel_data)), cmap='viridis')

In [ ]:
# Add channel labels
for _, row in channel_data.iterrows():
    ax.annotate(row['acquisition_channel'], 
                (row['cac'], row['lifetime_value']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

In [ ]:
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='black', linestyle='-', alpha=0.3)
ax.set_title('LTV vs CAC by Channel')
ax.set_xlabel('CAC (₹)')
ax.set_ylabel('LTV (₹)')
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/ltv_cac_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
4. FEATURE ENGINEERING FOR LTV PREDICTION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FEATURE ENGINEERING FOR LTV PREDICTION")
print("="*80)

In [ ]:
# Use first 30 days of behavior to predict LTV
def engineer_ltv_features(users_df, orders_df, ltv_df, lookback_days=30):
    """
    Engineer features for LTV prediction based on early behavior
    """
    
    print(f"\n🔄 Engineering features with {lookback_days}-day lookback...")
    
    # Create feature set
    features = users_df.copy()
    
    # For each user, calculate early behavior metrics
    early_metrics = []
    
    for user_id in features['user_id']:
        user_orders = orders_df[orders_df['user_id'] == user_id]
        
        # Orders in first N days
        if len(user_orders) > 0:
            first_order_date = user_orders['order_placed_at'].min()
            early_orders = user_orders[
                user_orders['order_placed_at'] <= first_order_date + timedelta(days=lookback_days)
            ]
        else:
            early_orders = pd.DataFrame()
        
        metrics = {
            'user_id': user_id,
            'early_order_count': len(early_orders),
            'early_total_spend': early_orders['total_amount'].sum() if len(early_orders) > 0 else 0,
            'early_avg_order_value': early_orders['total_amount'].mean() if len(early_orders) > 0 else 0,
            'early_unique_restaurants': early_orders['restaurant_id'].nunique() if len(early_orders) > 0 else 0,
            'early_cancellation_rate': (early_orders['order_status'] == 'cancelled').mean() if len(early_orders) > 0 else 0,
            'first_order_value': user_orders['total_amount'].min() if len(user_orders) > 0 else 0,
            'days_to_first_order': (user_orders['order_placed_at'].min() - features[features['user_id'] == user_id]['signup_date'].iloc[0]).days if len(user_orders) > 0 else 365
        }
        
        early_metrics.append(metrics)
    
    early_metrics_df = pd.DataFrame(early_metrics)
    features = features.merge(early_metrics_df, on='user_id', how='left')
    
    # Fill missing values
    numeric_cols = features.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        features[col] = features[col].fillna(0)
    
    # Add target variable (LTV)
    features = features.merge(
        ltv_df[['user_id', 'lifetime_value']], on='user_id', how='left'
    )
    
    # Filter users who have LTV (at least one order)
    features = features[features['lifetime_value'].notna()]
    
    return features

In [ ]:
# Create features for LTV prediction
ltv_features = engineer_ltv_features(users, delivered_orders, ltv_data, lookback_days=30)

In [ ]:
print(f"\n✅ Features engineered for {len(ltv_features):,} users")
print(f"📊 Features shape: {ltv_features.shape}")

---------------------------------------------------------------------
5. TRAIN LTV PREDICTION MODEL
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("TRAIN LTV PREDICTION MODEL")
print("="*80)

In [ ]:
# Select features for modeling
feature_columns = [
    'early_order_count', 'early_total_spend', 'early_avg_order_value',
    'early_unique_restaurants', 'early_cancellation_rate', 'first_order_value',
    'days_to_first_order', 'is_premium_member'
]

In [ ]:
# Handle categorical features
categorical_cols = ['acquisition_channel', 'age_band', 'device_type']

In [ ]:
# Prepare X and y
X = ltv_features[feature_columns].copy()
y = ltv_features['lifetime_value'].copy()

In [ ]:
# Encode categorical variables
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    ltv_features[col + '_encoded'] = le.fit_transform(ltv_features[col].fillna('unknown'))
    X[col + '_encoded'] = ltv_features[col + '_encoded']
    label_encoders[col] = le

In [ ]:
print(f"✅ Data prepared: X shape = {X.shape}, y shape = {y.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
print(f"📊 Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---------------------------------------------------------------------
6. MODEL TRAINING AND EVALUATION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("MODEL TRAINING AND EVALUATION")
print("="*80)

In [ ]:
# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.01),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

In [ ]:
# Train and evaluate models
results = []

In [ ]:
for name, model in models.items():
    print(f"\n🔄 Training {name}...")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Calculate MAPE (handle zeros)
    mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1))) * 100
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2,
        'MAPE (%)': mape
    })
    
    print(f"  MAE: ₹{mae:.2f}")
    print(f"  RMSE: ₹{rmse:.2f}")
    print(f"  R²: {r2:.4f}")
    print(f"  MAPE: {mape:.2f}%")

In [ ]:
# Results summary
results_df = pd.DataFrame(results)
print("\n📊 Model Comparison:")
print(results_df.to_string(index=False))

In [ ]:
# Feature importance for best model
best_model_name = results_df.loc[results_df['R²'].idxmax(), 'Model']
best_model = models[best_model_name]

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\n📊 {best_model_name} Feature Importance:")
    print(feature_importance.head(10).to_string(index=False))
    
    # Visualize feature importance
    fig, ax = plt.subplots(figsize=(10, 8))
    top_features = feature_importance.head(15)
    ax.barh(top_features['feature'], top_features['importance'], color='#3498db', alpha=0.7)
    ax.set_title(f'Top 15 Features for LTV Prediction ({best_model_name})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('../outputs/visualizations/ltv_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
elif hasattr(best_model, 'coef_'):
    coefficients = pd.DataFrame({
        'feature': X.columns,
        'coefficient': best_model.coef_
    }).sort_values('coefficient', ascending=False)
    
    print(f"\n📊 {best_model_name} Coefficients:")
    print(coefficients.head(10).to_string(index=False))

---------------------------------------------------------------------
7. PREDICT LTV FOR NEW USERS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("PREDICTING LTV FOR NEW USERS")
print("="*80)

In [ ]:
# Use best model for prediction
best_model = models[results_df.loc[results_df['R²'].idxmax(), 'Model']]

In [ ]:
# Predict LTV for all users
X_all_scaled = scaler.transform(X)
ltv_features['predicted_ltv'] = best_model.predict(X_all_scaled)

In [ ]:
# Calculate prediction accuracy
ltv_features['ltv_ratio'] = ltv_features['predicted_ltv'] / ltv_features['lifetime_value']
ltv_features['ltv_ratio'] = ltv_features['ltv_ratio'].clip(upper=5)  # Cap extreme values

In [ ]:
print(f"\n📊 Predicted LTV Statistics:")
print(ltv_features['predicted_ltv'].describe())

In [ ]:
# Compare predicted vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('LTV Prediction Performance', fontsize=14, fontweight='bold')

In [ ]:
# Predicted vs Actual
ax = axes[0]
sample = ltv_features.sample(1000)
ax.scatter(sample['lifetime_value'], sample['predicted_ltv'], alpha=0.5, color='#3498db')
ax.plot([0, sample['lifetime_value'].max()], [0, sample['lifetime_value'].max()], 
        'r--', label='Perfect Prediction')
ax.set_xlabel('Actual LTV (₹)')
ax.set_ylabel('Predicted LTV (₹)')
ax.set_title('Predicted vs Actual LTV')
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
# Prediction error distribution
ax = axes[1]
errors = ltv_features['predicted_ltv'] - ltv_features['lifetime_value']
errors.hist(bins=50, ax=ax, color='#2ecc71', alpha=0.7, edgecolor='black')
ax.axvline(0, color='red', linestyle='--', label='Perfect Prediction')
ax.set_title('Prediction Error Distribution')
ax.set_xlabel('Error (₹)')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/ltv_prediction_performance.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
8. LTV SEGMENTATION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("LTV SEGMENTATION")
print("="*80)

In [ ]:
# Create LTV segments
ltv_features['ltv_segment'] = pd.qcut(
    ltv_features['lifetime_value'],
    q=4,
    labels=['Low', 'Medium-Low', 'Medium-High', 'High']
)

In [ ]:
print("\n📊 LTV Segment Distribution:")
ltv_segment_counts = ltv_features['ltv_segment'].value_counts().sort_index()
for segment, count in ltv_segment_counts.items():
    pct = count / len(ltv_features) * 100
    print(f"  • {segment}: {count:,} users ({pct:.1f}%)")

In [ ]:
# Analyze segment characteristics
segment_analysis = ltv_features.groupby('ltv_segment').agg({
    'lifetime_value': ['mean', 'median', 'std'],
    'order_count': 'mean',
    'avg_order_value': 'mean',
    'early_order_count': 'mean',
    'early_total_spend': 'mean',
    'is_premium_member': 'mean'
}).round(2)

In [ ]:
print("\n📊 Segment Characteristics:")
print(segment_analysis)

In [ ]:
# LTV segment distribution by acquisition channel
segment_channel = pd.crosstab(
    ltv_features['ltv_segment'], 
    ltv_features['acquisition_channel'], 
    normalize='index'
) * 100

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
segment_channel.plot(kind='bar', ax=ax, stacked=True)
ax.set_title('LTV Segment Distribution by Acquisition Channel', fontsize=14, fontweight='bold')
ax.set_xlabel('LTV Segment')
ax.set_ylabel('Percentage of Users')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../outputs/visualizations/ltv_segment_channel.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
9. ACQUISITION OPTIMIZATION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("ACQUISITION OPTIMIZATION RECOMMENDATIONS")
print("="*80)

In [ ]:
# Calculate expected LTV by channel
channel_ltv_pred = ltv_features.groupby('acquisition_channel')['predicted_ltv'].mean()
channel_cac = users.groupby('acquisition_channel')['signup_channel_cost'].mean().fillna(0)

In [ ]:
channel_optimization = pd.DataFrame({
    'channel': channel_ltv_pred.index,
    'predicted_ltv': channel_ltv_pred.values,
    'cac': [channel_cac.get(ch, 0) for ch in channel_ltv_pred.index]
})

In [ ]:
channel_optimization['ltv_cac_ratio'] = channel_optimization['predicted_ltv'] / (channel_optimization['cac'] + 1)
channel_optimization['optimization_potential'] = channel_optimization['ltv_cac_ratio'] - 1

In [ ]:
print("\n📊 Acquisition Optimization Analysis:")
print(channel_optimization.to_string(index=False))

In [ ]:
# Calculate optimal spend allocation
total_budget = 1000000  # ₹10L budget
channel_optimization['recommended_allocation'] = (
    channel_optimization['ltv_cac_ratio'] / channel_optimization['ltv_cac_ratio'].sum() * total_budget
)
channel_optimization['recommended_allocation'] = channel_optimization['recommended_allocation'].round(0)

In [ ]:
print("\n💰 Recommended Budget Allocation:")
print(channel_optimization[['channel', 'recommended_allocation', 'ltv_cac_ratio']].to_string(index=False))

In [ ]:
# Visualize optimization
fig, ax = plt.subplots(figsize=(12, 6))

In [ ]:
channels = channel_optimization['channel']
ratios = channel_optimization['ltv_cac_ratio']
colors = ['#2ecc71' if x > 3 else '#f39c12' if x > 1 else '#e74c3c' for x in ratios]

In [ ]:
bars = ax.bar(channels, ratios, color=colors, alpha=0.7)
ax.axhline(y=3, color='green', linestyle='--', label='Good (3x)', linewidth=2)
ax.axhline(y=1, color='red', linestyle='--', label='Breakeven (1x)', linewidth=2)
ax.set_title('LTV:CAC Ratio by Channel with Optimization Recommendations', fontsize=14, fontweight='bold')
ax.set_xlabel('Acquisition Channel')
ax.set_ylabel('LTV:CAC Ratio')
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
# Add budget allocation labels
for bar, ratio, budget in zip(bars, ratios, channel_optimization['recommended_allocation']):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.1,
            f'₹{budget:,.0f}', ha='center', va='bottom', fontsize=8, rotation=0)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/acquisition_optimization.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
10. LTV PREDICTION FOR PORTFOLIO
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("LTV PREDICTION DASHBOARD")
print("="*80)

In [ ]:
# Create LTV dashboard
dashboard_data = {
    'Metric': [
        'Total Users with LTV',
        'Average LTV',
        'Median LTV',
        'LTV Range',
        'Top 10% LTV Threshold',
        'Total LTV of All Users',
        'LTV:CAC Ratio (Overall)',
        'Best Channel LTV',
        'Best Channel CAC',
        'Prediction Model R²',
        'Prediction Model MAE',
        'High LTV Users (%)'
    ],
    'Value': [
        f"{len(ltv_features):,}",
        f"₹{ltv_features['lifetime_value'].mean():,.2f}",
        f"₹{ltv_features['lifetime_value'].median():,.2f}",
        f"₹{ltv_features['lifetime_value'].min():,.2f} - ₹{ltv_features['lifetime_value'].max():,.2f}",
        f"₹{ltv_features['lifetime_value'].quantile(0.9):,.2f}",
        f"₹{ltv_features['lifetime_value'].sum():,.2f}",
        f"{ltv_features['lifetime_value'].mean() / users['signup_channel_cost'].mean():.2f}x",
        f"{channel_ltv_pred.max():.2f}",
        f"₹{channel_cac.min():.2f}",
        f"{results_df[results_df['Model'] == best_model_name]['R²'].values[0]:.3f}",
        f"₹{results_df[results_df['Model'] == best_model_name]['MAE'].values[0]:.2f}",
        f"{(ltv_features['ltv_segment'] == 'High').mean() * 100:.1f}%"
    ]
}

In [ ]:
dashboard_df = pd.DataFrame(dashboard_data)
print("\n📊 LTV Prediction Dashboard:")
print(dashboard_df.to_string(index=False))

---------------------------------------------------------------------
11. BUSINESS RECOMMENDATIONS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("BUSINESS RECOMMENDATIONS")
print("="*80)

In [ ]:
print("""
🏆 KEY INSIGHTS & RECOMMENDATIONS:
==================================

1. LTV INSIGHTS:
   • Average LTV: ₹{avg_ltv:,.0f}
   • Median LTV: ₹{med_ltv:,.0f}
   • Top 10% users drive {top_pct_revenue:.1f}% of total LTV
   • High LTV users: {(ltv_features['ltv_segment'] == 'High').sum():,} users

2. CHANNEL PERFORMANCE:
   • Best channel by LTV:CAC: {best_channel}
   • Worst channel by LTV:CAC: {worst_channel}
   • Opportunity: Reallocate ₹{reallocation_amount:,.0f} to high-performing channels

3. PREDICTION MODEL:
   • Best performing: {best_model_name}
   • R² Score: {best_r2:.3f}
   • MAE: ₹{best_mae:.2f}
   • Can predict LTV with {best_mape:.1f}% accuracy

4. SEGMENT OPPORTUNITIES:
   • High LTV users are {high_ltv_premium:.1f}x more likely to be premium members
   • Referral users have {referral_ltv_ratio:.1f}x higher LTV than paid channels
   • Early behavior (first 30 days) explains {early_ltv_corr:.2f} of LTV variance

🎯 ACTIONABLE RECOMMENDATIONS:
==============================

PRIORITY 1 (Immediate - Next 30 Days):
---------------------------------------
1. Optimize acquisition spend:
   • Increase budget for {best_channels} channels
   • Reduce spend on {worst_channels} channels
   • Target users likely to become high LTV

2. Implement early LTV prediction:
   • Identify high-potential users in first 30 days
   • Focus retention efforts on predicted high-LTV users
   • Customize onboarding for high-potential segments

3. Improve LTV for low-value segments:
   • Upsell premium membership to medium-LTV users
   • Cross-sell campaigns for low-LTV users
   • Referral program to acquire high-LTV users

PRIORITY 2 (Short-term - Next 90 Days):
--------------------------------------
1. Develop segment-specific strategies:
   • High LTV: Loyalty program, exclusive offers
   • Medium LTV: Engagement campaigns, premium upsell
   • Low LTV: Win-back campaigns, reactivation offers

2. Optimize acquisition channels:
   • A/B test new creative for high-LTV segments
   • Measure incremental LTV from each channel
   • Implement automated bid optimization

3. Build LTV prediction pipeline:
   • Real-time LTV scoring for new users
   • Integration with marketing automation
   • Continuous model retraining

PRIORITY 3 (Long-term - Next 6 Months):
--------------------------------------
1. Develop predictive LTV model v2:
   • Incorporate more features (engagement, support tickets)
   • Use survival analysis for better predictions
   • Implement ML pipeline for automated retraining

2. LTV-based personalization:
   • Differentiated user experiences based on LTV tier
   • Personalized offers and recommendations
   • Dynamic pricing based on LTV potential

📈 SUCCESS METRICS:
==================
• Increase average LTV by 15%
• Improve LTV:CAC ratio to 3:1
• Reduce acquisition cost by 20%
• Increase high-LTV users by 25%
""".format(
    avg_ltv=ltv_features['lifetime_value'].mean(),
    med_ltv=ltv_features['lifetime_value'].median(),
    top_pct_revenue=ltv_features[ltv_features['ltv_segment'] == 'High']['lifetime_value'].sum() / ltv_features['lifetime_value'].sum() * 100,
    best_channel=channel_optimization.iloc[channel_optimization['ltv_cac_ratio'].idxmax()]['channel'],
    worst_channel=channel_optimization.iloc[channel_optimization['ltv_cac_ratio'].idxmin()]['channel'],
    reallocation_amount=channel_optimization[channel_optimization['ltv_cac_ratio'] > 3]['recommended_allocation'].sum(),
    best_model_name=best_model_name,
    best_r2=results_df[results_df['Model'] == best_model_name]['R²'].values[0],
    best_mae=results_df[results_df['Model'] == best_model_name]['MAE'].values[0],
    best_mape=results_df[results_df['Model'] == best_model_name]['MAPE (%)'].values[0],
    high_ltv_premium=ltv_features[ltv_features['ltv_segment'] == 'High']['is_premium_member'].mean() / ltv_features['is_premium_member'].mean(),
    referral_ltv_ratio=ltv_features[ltv_features['acquisition_channel'] == 'referral']['lifetime_value'].mean() / ltv_features[ltv_features['acquisition_channel'] != 'referral']['lifetime_value'].mean(),
    early_ltv_corr=ltv_features['early_total_spend'].corr(ltv_features['lifetime_value']),
    best_channels=', '.join(channel_optimization[channel_optimization['ltv_cac_ratio'] > 3]['channel'].tolist()),
    worst_channels=', '.join(channel_optimization[channel_optimization['ltv_cac_ratio'] < 1]['channel'].tolist())
))

---------------------------------------------------------------------
12. EXPORT RESULTS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPORTING RESULTS")
print("="*80)

In [ ]:
# Save LTV data
ltv_features.to_csv('../outputs/cleaned_data/ltv_predictions.csv', index=False)

In [ ]:
# Save channel optimization
channel_optimization.to_csv('../outputs/cleaned_data/channel_optimization.csv', index=False)

In [ ]:
# Save model results
results_df.to_csv('../outputs/cleaned_data/ltv_model_results.csv', index=False)

In [ ]:
# Save LTV dashboard
dashboard_df.to_csv('../outputs/cleaned_data/ltv_dashboard.csv', index=False)

In [ ]:
print("✅ LTV predictions saved to ../outputs/cleaned_data/ltv_predictions.csv")
print("✅ Channel optimization saved to ../outputs/cleaned_data/channel_optimization.csv")
print("✅ Model results saved to ../outputs/cleaned_data/ltv_model_results.csv")
print("✅ Dashboard saved to ../outputs/cleaned_data/ltv_dashboard.csv")

---------------------------------------------------------------------
13. FINAL